In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd()

MODEL_DIR = PROJECT_ROOT / "models"
CWRU_DIR = PROJECT_ROOT / "data" / "processed" / "CWRU"
WINDOW_DIR = CWRU_DIR / "windows"

MODEL_PATH = MODEL_DIR / "vaac_tiny_best.keras"

model = tf.keras.models.load_model(
    MODEL_PATH
)

print("Model:", MODEL_PATH)
print("Input:", model.input_shape)
print("Output:", model.output_shape)
print("Parameters:", model.count_params())

Model: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\models\vaac_tiny_best.keras
Input: (None, 12000, 1)
Output: (None, 4)
Parameters: 4244


In [3]:
TEST_METADATA = WINDOW_DIR / "test_metadata.csv"

test_df = pd.read_csv(TEST_METADATA)

print("Test windows:", len(test_df))

print("\nTest classes:")
print(test_df["class"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts().sort_index())

Test windows: 136

Test classes:
class
Healthy       79
Ball          19
Inner Race    19
Outer Race    19
Name: count, dtype: int64

Test labels:
label
0    79
1    19
2    19
3    19
Name: count, dtype: int64


In [4]:
all_npy_files = list(
    CWRU_DIR.rglob("*.npy")
)

window_file_index = {
    path.name: path
    for path in all_npy_files
}

print(
    "Indexed .npy files:",
    len(window_file_index)
)

Indexed .npy files: 583


In [5]:
def get_window_path(row):

    filename = (
        f"{row['recording_id']}"
        f"_window_{int(row['window_id']):04d}.npy"
    )

    if filename not in window_file_index:
        raise FileNotFoundError(
            f"Window not found: {filename}"
        )

    return window_file_index[filename]

In [6]:
row = test_df.iloc[0]

path = get_window_path(row)

signal = np.load(path).astype(np.float32)

print("Class:", row["class"])
print("Label:", row["label"])
print("Recording:", row["recording_id"])
print("Window:", row["window_id"])

print("\nShape:", signal.shape)
print("dtype:", signal.dtype)

print("Minimum:", signal.min())
print("Maximum:", signal.max())
print("Mean:", signal.mean())
print("Std:", signal.std())
print("RMS:", np.sqrt(np.mean(signal ** 2)))

Class: Ball
Label: 1
Recording: B007_3_X121
Window: 0

Shape: (12000,)
dtype: float32
Minimum: -0.609619
Maximum: 0.62391335
Mean: 0.00342818
Std: 0.15476662
RMS: 0.15480459


In [7]:
print("Raw signal statistics")
print("---------------------")

print("Min :", float(signal.min()))
print("Max :", float(signal.max()))
print("Mean:", float(signal.mean()))
print("Std :", float(signal.std()))

Raw signal statistics
---------------------
Min : -0.6096190214157104
Max : 0.6239133477210999
Mean: 0.0034281800035387278
Std : 0.15476661920547485


In [8]:
raw_signal = signal.copy()

In [9]:
standardized_signal = (
    signal - signal.mean()
) / (
    signal.std() + 1e-8
)

In [10]:
max_abs_signal = (
    signal /
    (np.max(np.abs(signal)) + 1e-8)
)

In [11]:
raw_input = raw_signal.reshape(
    1, 12000, 1
)

raw_output = model.predict(
    raw_input,
    verbose=0
)[0]

print("RAW INPUT")
print("Output:", raw_output)
print("Predicted label:", np.argmax(raw_output))
print("Confidence:", np.max(raw_output))
print("True label:", row["label"])

RAW INPUT
Output: [1.2437244e-04 8.3114700e-03 3.2581473e-05 9.9153155e-01]
Predicted label: 3
Confidence: 0.99153155
True label: 1


In [12]:
standardized_input = (
    standardized_signal
    .reshape(1, 12000, 1)
)

standardized_output = model.predict(
    standardized_input,
    verbose=0
)[0]

print("STANDARDIZED INPUT")
print("Output:", standardized_output)
print(
    "Predicted label:",
    np.argmax(standardized_output)
)
print(
    "Confidence:",
    np.max(standardized_output)
)
print("True label:", row["label"])

STANDARDIZED INPUT
Output: [2.7167296e-06 9.9199486e-01 7.9985056e-03 3.8848684e-06]
Predicted label: 1
Confidence: 0.99199486
True label: 1


In [13]:
max_abs_input = (
    max_abs_signal
    .reshape(1, 12000, 1)
)

max_abs_output = model.predict(
    max_abs_input,
    verbose=0
)[0]

print("MAX-ABS NORMALIZED INPUT")
print("Output:", max_abs_output)
print(
    "Predicted label:",
    np.argmax(max_abs_output)
)
print(
    "Confidence:",
    np.max(max_abs_output)
)
print("True label:", row["label"])

MAX-ABS NORMALIZED INPUT
Output: [6.5533502e-05 1.8525906e-02 2.9252324e-05 9.8137939e-01]
Predicted label: 3
Confidence: 0.9813794
True label: 1


In [14]:
print("=" * 60)
print("INPUT PREPROCESSING COMPARISON")
print("=" * 60)

print(
    "True label:",
    row["label"]
)

print(
    "\nRAW:",
    np.argmax(raw_output),
    "confidence:",
    float(np.max(raw_output))
)

print(
    "STANDARDIZED:",
    np.argmax(standardized_output),
    "confidence:",
    float(np.max(standardized_output))
)

print(
    "MAX-ABS:",
    np.argmax(max_abs_output),
    "confidence:",
    float(np.max(max_abs_output))
)

INPUT PREPROCESSING COMPARISON
True label: 1

RAW: 3 confidence: 0.9915315508842468
STANDARDIZED: 1 confidence: 0.9919948577880859
MAX-ABS: 3 confidence: 0.9813793897628784


In [15]:
print(
    model.get_config()
)

{'name': 'VAAC_Tiny_CNN', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': (None, 12000, 1), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'vibration_input'}, 'registered_name': None, 'name': 'vibration_input', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'Conv1D', 'config': {'name': 'initial_conv', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'filters': 16, 'kernel_size': (7,), 'strides': (1,), 'padding': 'same', 'data_format': 'channels_last', 'dilation_rate': (1,), 'groups': 1, 'activation': 'relu', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_r

In [16]:
print("\nMODEL LAYERS")

for layer in model.layers:

    print(
        layer.name,
        "->",
        layer.__class__.__name__,
        "->",
        layer.output_shape
        if hasattr(layer, "output_shape")
        else "N/A"
    )


MODEL LAYERS
vibration_input -> InputLayer -> N/A
initial_conv -> Conv1D -> N/A
scale_3 -> SeparableConv1D -> N/A
scale_7 -> SeparableConv1D -> N/A
scale_15 -> SeparableConv1D -> N/A
multi_scale_fusion -> Concatenate -> N/A
feature_refinement -> SeparableConv1D -> N/A
global_average_pooling -> GlobalAveragePooling1D -> N/A
dense_features -> Dense -> N/A
classification_output -> Dense -> N/A


In [17]:
print(
    "Model input dtype:",
    model.inputs[0].dtype
)

Model input dtype: float32


In [18]:
results = []

for _, row in test_df.head(20).iterrows():

    path = get_window_path(row)

    signal = np.load(
        path
    ).astype(np.float32)

    signal = signal.reshape(
        1, 12000, 1
    )

    output = model.predict(
        signal,
        verbose=0
    )[0]

    prediction = int(
        np.argmax(output)
    )

    results.append({
        "class": row["class"],
        "true_label": int(row["label"]),
        "prediction": prediction,
        "confidence": float(np.max(output))
    })

diagnostic_df = pd.DataFrame(results)

display(diagnostic_df)

,class,true_label,prediction,confidence
0,Ball,1,3,0.991532
1,Ball,1,3,0.991519
2,Ball,1,3,0.991489
3,Ball,1,3,0.991643
4,Ball,1,3,0.991601
5,Ball,1,3,0.991387
6,Ball,1,3,0.991524
7,Ball,1,3,0.991909
8,Ball,1,3,0.992053
9,Ball,1,3,0.991958


In [19]:
diagnostic_accuracy = accuracy_score(
    diagnostic_df["true_label"],
    diagnostic_df["prediction"]
)

print(
    "Raw-input diagnostic accuracy:",
    f"{diagnostic_accuracy:.4f}"
)

Raw-input diagnostic accuracy: 0.0000


In [20]:
# STEP 227.15
# Full test evaluation using per-window standardization

y_true_std = []
y_pred_std = []
conf_std = []

for _, row in test_df.iterrows():

    path = get_window_path(row)

    signal = np.load(path).astype(np.float32)

    # Standardize exactly as diagnostic test
    signal_std = (
        signal - signal.mean()
    ) / (
        signal.std() + 1e-8
    )

    # Model input shape
    model_input = signal_std.reshape(
        1, 12000, 1
    )

    # Prediction
    output = model.predict(
        model_input,
        verbose=0
    )[0]

    prediction = int(
        np.argmax(output)
    )

    confidence = float(
        np.max(output)
    )

    y_true_std.append(
        int(row["label"])
    )

    y_pred_std.append(
        prediction
    )

    conf_std.append(
        confidence
    )

print("Standardized test evaluation completed.")
print("Test windows:", len(y_true_std))

Standardized test evaluation completed.
Test windows: 136


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

std_accuracy = accuracy_score(
    y_true_std,
    y_pred_std
)

std_precision = precision_score(
    y_true_std,
    y_pred_std,
    average="weighted",
    zero_division=0
)

std_recall = recall_score(
    y_true_std,
    y_pred_std,
    average="weighted",
    zero_division=0
)

std_f1 = f1_score(
    y_true_std,
    y_pred_std,
    average="weighted",
    zero_division=0
)

print("=" * 55)
print("STEP 227 — STANDARDIZED FP32 TEST PERFORMANCE")
print("=" * 55)

print(f"Accuracy : {std_accuracy:.4f}")
print(f"Precision: {std_precision:.4f}")
print(f"Recall   : {std_recall:.4f}")
print(f"F1-score : {std_f1:.4f}")

STEP 227 — STANDARDIZED FP32 TEST PERFORMANCE
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-score : 1.0000


In [22]:
from sklearn.metrics import confusion_matrix

cm_std = confusion_matrix(
    y_true_std,
    y_pred_std,
    labels=[0, 1, 2, 3]
)

print("STANDARDIZED FP32 CONFUSION MATRIX")
print("=" * 55)
print(cm_std)

STANDARDIZED FP32 CONFUSION MATRIX
[[79  0  0  0]
 [ 0 19  0  0]
 [ 0  0 19  0]
 [ 0  0  0 19]]


In [23]:
from sklearn.metrics import classification_report

CLASS_NAMES = [
    "Healthy",
    "Ball",
    "Inner Race",
    "Outer Race"
]

print(
    classification_report(
        y_true_std,
        y_pred_std,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

              precision    recall  f1-score   support

     Healthy       1.00      1.00      1.00        79
        Ball       1.00      1.00      1.00        19
  Inner Race       1.00      1.00      1.00        19
  Outer Race       1.00      1.00      1.00        19

    accuracy                           1.00       136
   macro avg       1.00      1.00      1.00       136
weighted avg       1.00      1.00      1.00       136



In [24]:
print("TRUE LABEL COUNTS")
print(
    pd.Series(y_true_std)
    .value_counts()
    .sort_index()
)

print("\nPREDICTED LABEL COUNTS")
print(
    pd.Series(y_pred_std)
    .value_counts()
    .sort_index()
)

TRUE LABEL COUNTS
0    79
1    19
2    19
3    19
Name: count, dtype: int64

PREDICTED LABEL COUNTS
0    79
1    19
2    19
3    19
Name: count, dtype: int64


In [25]:
conf_std = np.array(conf_std)

print("=" * 55)
print("STANDARDIZED FP32 CONFIDENCE")
print("=" * 55)

print(
    "Mean:",
    float(conf_std.mean())
)

print(
    "Minimum:",
    float(conf_std.min())
)

print(
    "Maximum:",
    float(conf_std.max())
)

STANDARDIZED FP32 CONFIDENCE
Mean: 0.9959809929132462
Minimum: 0.959564208984375
Maximum: 0.9999984502792358


In [26]:
raw_accuracy = accuracy_score(
    diagnostic_df["true_label"],
    diagnostic_df["prediction"]
)

print("=" * 65)
print("STEP 227 — PREPROCESSING EFFECT")
print("=" * 65)

print(
    f"RAW accuracy "
    f"(20-window diagnostic): "
    f"{raw_accuracy:.4f}"
)

print(
    f"STANDARDIZED accuracy "
    f"(full test): "
    f"{std_accuracy:.4f}"
)

print("\nStandardization formula:")
print("x_std = (x - mean(x)) / (std(x) + 1e-8)")

STEP 227 — PREPROCESSING EFFECT
RAW accuracy (20-window diagnostic): 0.0000
STANDARDIZED accuracy (full test): 1.0000

Standardization formula:
x_std = (x - mean(x)) / (std(x) + 1e-8)


In [27]:
raw_true = []
raw_pred = []

standardized_true = []
standardized_pred = []

for _, row in test_df.iterrows():

    path = get_window_path(row)

    signal = np.load(
        path
    ).astype(np.float32)

    true_label = int(row["label"])

    # -------------------------
    # RAW
    # -------------------------
    raw_input = signal.reshape(
        1, 12000, 1
    )

    raw_output = model.predict(
        raw_input,
        verbose=0
    )[0]

    raw_prediction = int(
        np.argmax(raw_output)
    )

    # -------------------------
    # STANDARDIZED
    # -------------------------
    signal_std = (
        signal - signal.mean()
    ) / (
        signal.std() + 1e-8
    )

    std_input = signal_std.reshape(
        1, 12000, 1
    )

    std_output = model.predict(
        std_input,
        verbose=0
    )[0]

    std_prediction = int(
        np.argmax(std_output)
    )

    raw_true.append(true_label)
    raw_pred.append(raw_prediction)

    standardized_true.append(true_label)
    standardized_pred.append(std_prediction)

raw_full_accuracy = accuracy_score(
    raw_true,
    raw_pred
)

std_full_accuracy = accuracy_score(
    standardized_true,
    standardized_pred
)

print("=" * 65)
print("RAW vs STANDARDIZED — FULL 136 TEST WINDOWS")
print("=" * 65)

print(
    f"RAW accuracy          : "
    f"{raw_full_accuracy:.4f}"
)

print(
    f"STANDARDIZED accuracy : "
    f"{std_full_accuracy:.4f}"
)

print(
    f"Accuracy improvement  : "
    f"{std_full_accuracy - raw_full_accuracy:.4f}"
)

RAW vs STANDARDIZED — FULL 136 TEST WINDOWS
RAW accuracy          : 0.1397
STANDARDIZED accuracy : 1.0000
Accuracy improvement  : 0.8603


In [29]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd()

# Results directory
RESULTS_DIR = PROJECT_ROOT / "results"

# Create directory if it does not exist
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("RESULTS_DIR:")
print(RESULTS_DIR)

print("Directory exists:", RESULTS_DIR.exists())

RESULTS_DIR:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results
Directory exists: True


In [30]:
comparison_df = pd.DataFrame({
    "true_label": raw_true,
    "raw_prediction": raw_pred,
    "standardized_prediction": standardized_pred
})

comparison_path = (
    RESULTS_DIR /
    "vaac_tiny_raw_vs_standardized.csv"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print(
    "Saved:",
    comparison_path
)

Saved: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\vaac_tiny_raw_vs_standardized.csv


In [31]:
print("=" * 70)
print("STEP 227 — FP32 INPUT/PREPROCESSING VERIFICATION")
print("=" * 70)

print("\nModel: VAAC-Tiny")
print("Parameters:", model.count_params())

print("\nTest windows:", len(test_df))

print("\nRAW FP32 accuracy:")
print(f"{raw_full_accuracy:.4f}")

print("\nSTANDARDIZED FP32 accuracy:")
print(f"{std_full_accuracy:.4f}")

print("\nStandardization:")
print("(x - mean) / (std + 1e-8)")

if std_full_accuracy > raw_full_accuracy:
    print(
        "\nDiagnosis: Standardized input "
        "substantially improves model performance."
    )
else:
    print(
        "\nDiagnosis: Standardization did not "
        "improve performance."
    )

print("\nStep 227 completed.")

STEP 227 — FP32 INPUT/PREPROCESSING VERIFICATION

Model: VAAC-Tiny
Parameters: 4244

Test windows: 136

RAW FP32 accuracy:
0.1397

STANDARDIZED FP32 accuracy:
1.0000

Standardization:
(x - mean) / (std + 1e-8)

Diagnosis: Standardized input substantially improves model performance.

Step 227 completed.
